# Fast MF using eALS

This notebook is an implementation of eALS (element-wise ALS) for implicit feedback as described in *Fast Matrix Factorization for Online Recommendation with Implicit Feedback* (He et al., SIGIR 2016).

## Summary

1. Data preparation from raw review text (we chose the Amazon movies dataset).
2. eALS training with coordinate updates.
3. Offline ranking evaluation (HR@K, NDCG@K).
4. Saving artifacts for analysis/reuse.

This project separates logic into Python modules (`src/*`) and uses the notebook as an orchestrator.

In [1]:
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from src.data_ingest import prepare_data
from src.evaluate import evaluate_model
from src.train_eals import train_eals

REPO = Path(os.getcwd()).resolve()

DATA_DIR = REPO / "data"

DATASET = DATA_DIR / "AmazonMoviesDataset.txt"
if not DATASET.exists():
    raise FileNotFoundError(f"Expected dataset at: {DATASET}")

## Step 0. Choose Runtime Profile

There are two runtime modes:
- `smoke`: fast feedback loop for development and debugging, uses only a subset of the data.
- `full`: paper-aligned scale/configuration for real experiments.

In [2]:
MODE = "full"
MAX_SMOKE_RECORDS = 10000
if MODE.lower() not in {"full", "smoke"}:
    raise ValueError(f"Unsupported mode: {MODE}")

DATASET_SMOKE = DATA_DIR / "AmazonMoviesSmokeSubset.txt"

OUTPUT_DIR = REPO / "outputs" / MODE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if MODE == "smoke" and not DATASET_SMOKE.exists():
    records = 0
    with (
        DATASET.open("r", encoding="utf-8", errors="ignore") as fin,
        DATASET_SMOKE.open("w", encoding="utf-8") as fout,
    ):
        block = []
        for line in fin:
            if line.strip() == "":
                if block:
                    fout.writelines(block)
                    fout.write("\n")
                    records += 1
                    block = []
                    if records >= MAX_SMOKE_RECORDS:
                        break
                continue
            block.append(line)
        if records < MAX_SMOKE_RECORDS and block:
            fout.writelines(block)
            fout.write("\n")
            records += 1
    print(f"Created smoke subset with {records} records at {DATASET_SMOKE}")

DATASET_PATH = DATASET_SMOKE if MODE == "smoke" else DATASET
print("Dataset:", DATASET_PATH)
print("Output:", OUTPUT_DIR)

Dataset: /Users/Administrateur/School/X/Database/Project/fast-mf-for-recommendation/data/AmazonMoviesDataset.txt
Output: /Users/Administrateur/School/X/Database/Project/fast-mf-for-recommendation/outputs/full


## Step 1. Build eALS + Runtime Configs

This cell creates:
- `eals_config`: model/objective hyperparameters.
- `runtime_config`: Spark/runtime controls.

### Config from paper
- `factors=128`, `reg_lambda=0.01`, `c0=64`, `alpha=0.5`, `observed_weight=1`, `observed_value=1`, `iterations=100`, `topk=100`.

In [3]:
eals_config = {
    "factors": 128,
    "reg_lambda": 0.01,
    "c0": 64.0,
    "alpha": 0.5,
    "observed_weight": 1.0,
    "observed_value": 1.0,
    "iterations": 100,
    "topk": 100,
    "min_user_interactions": 10,
    "min_item_interactions": 10,
    "init_mean": 0.0,
    "init_std": 0.01,
    "random_seed": 42,
    "dtype": "float32",
}

runtime_config = {
    "mode": MODE,
    # Explicitly increase preprocessing parallelism to reduce per-task memory pressure.
    "num_partitions": 384,
    # Keep preprocessing spill-first for very large raw data stages.
    "preprocess_storage_level": "DISK_ONLY",
    # Keep training/eval histories in serialized form to reduce JVM heap footprint.
    "storage_level": "MEMORY_AND_DISK_SER",
    "eval_user_sample": None,
    "checkpoint_dir": None,
    "spark_log_level": "WARN",
}

print("eALS config:")
print(json.dumps(eals_config, indent=2))
print("\nRuntime config:")
print(json.dumps(runtime_config, indent=2))


eALS config:
{
  "factors": 128,
  "reg_lambda": 0.01,
  "c0": 64.0,
  "alpha": 0.5,
  "observed_weight": 1.0,
  "observed_value": 1.0,
  "iterations": 100,
  "topk": 100,
  "min_user_interactions": 10,
  "min_item_interactions": 10,
  "init_mean": 0.0,
  "init_std": 0.01,
  "random_seed": 42,
  "dtype": "float32"
}

Runtime config:
{
  "mode": "full",
  "num_partitions": 384,
  "preprocess_storage_level": "DISK_ONLY",
  "storage_level": "MEMORY_AND_DISK_SER",
  "eval_user_sample": null,
  "checkpoint_dir": null,
  "spark_log_level": "WARN"
}


## Step 2. Start Spark Session

The training/data prep pipeline is implemented with PySpark RDDs.
- Interactions and histories can be distributed.
- Factor matrices are broadcast for update passes (map-and-broadcast alternating updates).

In [4]:
spark = (
    SparkSession.builder.master("local[10]")
    .appName(f"eALS_notebook_{MODE}")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.driver.memory", "40g")
    .config("spark.executor.memory", "40g")
    .config("spark.python.worker.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.default.parallelism", "384")
    .config("spark.sql.shuffle.partitions", "384")
    .config("spark.driver.extraJavaOptions", "-XX:+UseG1GC")
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel(runtime_config["spark_log_level"])
print("Spark version:", spark.version)
print("Default parallelism:", sc.defaultParallelism)
print("Configured driver memory:", sc.getConf().get("spark.driver.memory"))
print("Configured partitions:", runtime_config["num_partitions"])


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/24 23:35:09 WARN Utils: Your hostname, Felix-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.131 instead (on interface en0)
26/03/24 23:35:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 23:35:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Default parallelism: 384
Configured driver memory: 40g
Configured partitions: 384


## Step 3. Prepare Data

This cell preprocesses the data.

1. Parse raw review blocks into `(user, item, timestamp)`.
2. Deduplicate repeated `(user, item)` pairs (keep latest timestamp).
3. Iterative user/item minimum-count filtering (k-core style).
4. Build contiguous integer indices for users/items.
5. Split by **chronological leave-one-out** per user (latest interaction as test).
6. Compute item-specific missing-data weights \(c_i\) using popularity (paper Eq. 8).
7. Build user/item histories needed by eALS updates.

In [5]:
prep_start = time.time()
prepared = prepare_data(
    sc=sc,
    dataset_path=str(DATASET_PATH),
    eals_config=eals_config,
    runtime_config=runtime_config,
    verbose=True,
)
prep_wall = time.time() - prep_start

if prepared.num_test_users == 0:
    raise RuntimeError(
        "No test users were produced after preprocessing. "
        "Increase smoke data size or relax min_user_interactions/min_item_interactions."
    )

print(f"Preparation wall time: {prep_wall:.2f} s")
stats_df = (
    pd.DataFrame(
        [{"stage": key, "value": value} for key, value in prepared.stats.items()]
    )
    .sort_values("stage")
    .reset_index(drop=True)
)
stats_df

[prepare] using preprocess storage level DISK_ONLY with 384 partitions


[prepare] loaded interactions: 7850072


[prepare] deduped interactions: 6770947


[prepare] k-core filtered interactions: 3768245


[prepare] indexed users/items: 110678/47202


[prepare] train interactions: 3657567, test users: 110678


[prepare] weighted items: 47201


[prepare] user/item histories: 110678/47201
Preparation wall time: 240.84 s


,stage,value
0,collect_ids_seconds,1.137227e+00
1,dedupe_seconds,6.189725e+00
2,deduped_interactions,6.770947e+06
3,filtered_interactions,3.768245e+06
4,history_seconds,1.584327e+01
5,index_seconds,2.995497e+01
6,indexed_interactions,3.768245e+06
7,item_histories,4.720100e+04
8,kcore_seconds,1.592468e+02
9,load_seconds,1.839538e+01


### See Data Structures

These are the sparse structures used to avoid iterating over the full user-item matrix.
- `user_history_rdd`: for each user, which items were observed and with what \(c_i\).
- `item_history_rdd`: for each item, which users observed it plus that item’s \(c_i\).

In [6]:
print("num_users:", prepared.num_users)
print("num_items:", prepared.num_items)
print("num_train_interactions:", prepared.num_train_interactions)
print("num_test_users:", prepared.num_test_users)

print("\nSample user history rows:")
for row in prepared.user_history_rdd.take(3):
    u, hist = row
    print("user", u, "history_len", len(hist), "first", hist[:3])

print("\nSample item history rows:")
for row in prepared.item_history_rdd.take(3):
    i, c_i, users = row
    print(
        "item",
        i,
        "c_i",
        round(float(c_i), 6),
        "users_len",
        len(users),
        "first",
        users[:5],
    )

num_users: 110678
num_items: 47202
num_train_interactions: 3657567
num_test_users: 110678

Sample user history rows:
user 37632 history_len 16 first [(13440, 0.0010044310134318128), (34632, 0.0010044310134318128), (2793, 0.0010044310134318128)]
user 65664 history_len 612 first [(16896, 0.002110212054838519), (46850, 0.0019827152026001727), (14211, 0.0018558322272344949)]
user 56448 history_len 90 first [(8832, 0.0023592888007792213), (24195, 0.002734893032390258), (24196, 0.002734893032390258)]

Sample item history rows:
item 13440 c_i 0.001004 users_len 29 first [37632, 103688, 38036, 48426, 83762]
item 16896 c_i 0.00211 users_len 128 first [65664, 15749, 21512, 18056, 66058]
item 8832 c_i 0.002359 users_len 160 first [56448, 5377, 26881, 97158, 36102]


## Step 3. Train eALS

Training alternates between:
- updating all user vectors `P` with item cache \(S^q\) (eq. 12 from the paper),
- updating all item vectors `Q` with user cache \(S^p\) (eq. 13 from the paper).
- Cache matrices implement the fast eALS idea that reduces repeated computation over missing entries.

The output `train_log` reports per-iteration times to see progress.

In [7]:
train_start = time.time()
model = train_eals(sc=sc, prepared_data=prepared, config=eals_config)
train_wall = time.time() - train_start
print(f"Training wall time: {train_wall:.2f} s")

train_log_df = pd.DataFrame(model.train_log)
train_log_df

Training wall time: 18593.36 s


,iteration,seconds
0,1,76.674400
1,2,75.474204
2,3,74.952651
3,4,74.871759
4,5,74.892456
...,...,...
95,96,75.054004
96,97,75.120694
97,98,75.343923
98,99,75.227867


## Step 4. Evaluate Ranking Quality

We evaluate recommendation quality with:
- `HR@K`: did the true held-out item appear in top-K?
- `NDCG@K`: was it ranked high in top-K?

With `K=100`, like in the paper.

In [8]:
eval_start = time.time()
metrics = evaluate_model(
    sc=sc,
    test_rdd=prepared.test_rdd,
    p_matrix=model.p_matrix,
    q_matrix=model.q_matrix,
    topk=eals_config["topk"],
    eval_user_sample=runtime_config["eval_user_sample"],
    random_seed=eals_config["random_seed"],
)
eval_wall = time.time() - eval_start
print(f"Evaluation wall time: {eval_wall:.2f} s")
metrics

Evaluation wall time: 27.91 s


{'hr': 0.7412132492455592,
 'ndcg': 0.6386674685247485,
 'evaluated_users': 110678}

## Step 5. Save Artifacts

- `P.npy`, `Q.npy`: learned latent factors.
- `metrics.json`: HR/NDCG summary.
- `prepare_stats.json`: preprocessing counters/timings.
- `train_log.json`: iteration timing.
- `id_maps.json`: mapping from internal indices to raw IDs.

In [9]:
np.save(OUTPUT_DIR / "P.npy", model.p_matrix)
np.save(OUTPUT_DIR / "Q.npy", model.q_matrix)
(OUTPUT_DIR / "metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "prepare_stats.json").write_text(
    json.dumps(prepared.stats, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "train_log.json").write_text(
    json.dumps({"iterations": model.train_log}, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "id_maps.json").write_text(
    json.dumps(
        {"user_ids": prepared.user_ids, "item_ids": prepared.item_ids}, indent=2
    ),
    encoding="utf-8",
)
print("Saved artifacts to", OUTPUT_DIR)

Saved artifacts to /Users/Administrateur/School/X/Database/Project/fast-mf-for-recommendation/outputs/full


## Step 6. Cleanup

Unpersist cached RDDs and stop Spark.

In [10]:
prepared.train_ui_rdd.unpersist()
prepared.test_rdd.unpersist()
prepared.user_history_rdd.unpersist()
prepared.item_history_rdd.unpersist()

spark.stop()
print("Spark stopped.")

Spark stopped.
